# Leakage Check & Unseen-Date Generalization Test

Built in response to Michel's feedback on `pcrtc/06`'s result (ZNCC=0.519):
"is this too good to be true?" Two independent checks, both against `06`'s
checkpoint (`s1_tuk_pcrtc_realattrs_unet_best.pth`, PC-RTC + real attrs):

1. **Spatial overlap check** -- structural proof that no validation patch
   shares physical ground with a training patch. Cheap, fast, direct.
2. **Unseen-date generalization test** -- behavioral evidence that the
   model works on Sentinel-1 imagery it has never touched at all, not
   just unseen patches from the same 7 training acquisitions. Fetches new
   data from Microsoft Planetary Computer (`sentinel-1-rtc`), one year
   after the original survey date so season/ice conditions stay
   comparable. Ground truth is the same LiDAR survey used throughout the
   project -- terrain roughness doesn't change, only the input imagery is
   new.

Together: step 1 rules out leakage in the *setup*, step 2 rules out
memorization in the *result*. Neither alone is sufficient on its own.

## Setup

In [1]:
import os
import sys
import json
import random
import datetime as dt
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds, transform as win_transform
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), 'CUDA is required. Run this notebook on the GPU environment.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 4070 Ti SUPER


In [2]:
WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
TESSA_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche/tessa_baseline')
REGION = 'tuk'
CHECKPOINT_NAME = 's1_tuk_pcrtc_realattrs_unet_best.pth'  # pcrtc/06's checkpoint -- best result of the study
S1_DIR = WORKING_REPO / 'input_data' / 's1_patches_tuk_pcrtc'
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_tuk_tessa'
CHECKPOINT_DIR = WORKING_REPO / 'checkpoints'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LIDAR_SURVEY_DATE = dt.date(2024, 4, 16)
CONTEXT_K = 3
TARGET_HW = (256, 256)
TIMESTEPS = 1000
VAL_FRACTION = 0.15
SEED = 42
NOISE_SCHEDULE = 'linear'
ATTENTION_VARIANT = 'default'

In [3]:
sys.path.insert(0, str(TESSA_REPO))
from src.model.unet import ConditionalUNet
from src.diffusion.scheduler import LinearDiffusionScheduler, CosineDiffusionScheduler
from src.diffusion.sampling import p_sample_loop_ddim
from src.utils.recon_metrics import rmse, bias, sigma_error, normal_angle_error, average_jsd_multiscale, log_psd_rmse, zncc

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

In [4]:
def build_real_attrs(s1_path, times, context_k):
    attrs_path = s1_path / 'attrs.json'
    attrs_list = json.load(open(attrs_path)) if attrs_path.exists() else []
    vecs = []
    for time_path in times:
        idx = int(time_path.stem[1:])
        a = attrs_list[idx] if idx < len(attrs_list) else {}
        if a.get('acquisition_date'):
            acq_date = dt.date.fromisoformat(a['acquisition_date'])
            age_days = (acq_date - LIDAR_SURVEY_DATE).days
            age_norm = age_days / 30.0
        else:
            age_norm = 0.0
        orbit_dir = 1.0 if a.get('orbit_direction') == 'ASCENDING' else 0.0
        rel_orbit = (a.get('relative_orbit_number') or 0) / 175.0
        vecs.append([age_norm, orbit_dir, rel_orbit, 0.0, 0.0, 0.0, 0.0, 0.0])
    return torch.tensor(vecs, dtype=torch.float32).flatten()


class LidarS1Dataset(Dataset):
    def __init__(self, s1_dir, lidar_dir, patch_ids, context_k=3, target_hw=(256, 256)):
        self.s1_dir = Path(s1_dir)
        self.lidar_dir = Path(lidar_dir)
        self.patch_ids = list(patch_ids)
        self.context_k = context_k
        self.target_hw = target_hw

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, index):
        patch_id = self.patch_ids[index]
        lidar_path = self.lidar_dir / f'lidar_patch_{patch_id}.tif'
        s1_path = self.s1_dir / f's1_patch_{patch_id}'
        with rasterio.open(lidar_path) as src:
            raw = src.read().astype(np.float32)
        target = raw[0]
        mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(target)
        target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
        valid_count = max(1, int(mask.sum()))
        patch_mean = float(target[mask].sum() / valid_count)
        target = (target - patch_mean) * mask

        times = sorted(s1_path.glob('t*.tif'))[:self.context_k]
        if len(times) < self.context_k:
            raise RuntimeError(f'{s1_path} has fewer than {self.context_k} Sentinel-1 times')
        views = []
        for time_path in times:
            with rasterio.open(time_path) as src:
                sar = src.read()[:2].astype(np.float32)
            sar = np.nan_to_num(sar, nan=0.0, posinf=0.0, neginf=0.0)
            sar = np.maximum(sar, 1e-12)
            sar = 10.0 * np.log10(sar)
            sar_tensor = torch.from_numpy(sar).unsqueeze(0)
            sar_tensor = F.interpolate(sar_tensor, size=self.target_hw, mode='bilinear', align_corners=False).squeeze(0)
            sar_tensor = sar_tensor.repeat(2, 1, 1)
            views.append(sar_tensor)
        condition = torch.cat(views, dim=0)
        attrs = build_real_attrs(s1_path, times, self.context_k)
        return {'lidar': torch.from_numpy(target).unsqueeze(0).float(), 'mask': torch.from_numpy(mask),
                's1': condition.float(), 'attrs': attrs, 'patch_mean': torch.tensor(patch_mean), 'patch_id': patch_id}

In [5]:
lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
s1_ids = {p.name.split('_')[-1] for p in S1_DIR.glob('s1_patch_*') if p.is_dir()}
paired_ids = sorted(lidar_ids & s1_ids)
assert paired_ids, 'No paired Sentinel-1/LiDAR patches found.'
random.Random(SEED).shuffle(paired_ids)
n_val = max(1, int(len(paired_ids) * VAL_FRACTION))
val_ids, train_ids = paired_ids[:n_val], paired_ids[n_val:]
print(f'Paired: {len(paired_ids)} | train: {len(train_ids)} | validation: {len(val_ids)}')

model = ConditionalUNet(in_channels=1, cond_channels=4 * CONTEXT_K, attr_dim=8 * CONTEXT_K, base_channels=128,
                         embed_dim=256, unet_depth=4, attention_variant=ATTENTION_VARIANT, cond_k=CONTEXT_K).to(DEVICE)
scheduler = LinearDiffusionScheduler(TIMESTEPS, device=DEVICE) if NOISE_SCHEDULE == 'linear' else CosineDiffusionScheduler(TIMESTEPS, device=DEVICE)
checkpoint = torch.load(CHECKPOINT_DIR / CHECKPOINT_NAME, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
sampler = p_sample_loop_ddim
print('Loaded checkpoint:', CHECKPOINT_NAME, '| epoch:', checkpoint.get('epoch'), '| val_loss:', checkpoint.get('val_loss'))

Paired: 1676 | train: 1425 | validation: 251
Loaded checkpoint: s1_tuk_pcrtc_realattrs_unet_best.pth | epoch: 100 | val_loss: 0.010252635416691191


## Step 1 -- Spatial overlap check

Does any validation patch share physical ground with a training patch?
This is the direct, structural test -- if it prints `0 / N`, the split is
provably clean at the geometry level, not just at the patch-ID level
(patch IDs are already guaranteed disjoint by construction; this checks
something the ID split alone can't).

In [6]:
from shapely.geometry import box
from shapely.strtree import STRtree

train_boxes = []
for pid in train_ids:
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src:
        train_boxes.append(box(*src.bounds))

val_boxes = []
for pid in val_ids:
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src:
        val_boxes.append((pid, box(*src.bounds)))

tree = STRtree(train_boxes)
overlap_count = 0
overlap_details = []
for pid, vbox in val_boxes:
    hits = tree.query(vbox)
    real_overlaps = [h for h in hits if train_boxes[h].intersects(vbox) and not train_boxes[h].touches(vbox)]
    if real_overlaps:
        overlap_count += 1
        overlap_details.append({'val_patch_id': pid, 'n_overlapping_train_patches': len(real_overlaps)})
        print(f'Val patch {pid} overlaps {len(real_overlaps)} training patch(es)')

print(f'\n{overlap_count} / {len(val_ids)} validation patches overlap a training patch')

overlap_result = {
    'checkpoint': CHECKPOINT_NAME,
    'n_train': len(train_ids), 'n_val': len(val_ids),
    'n_val_patches_overlapping_train': overlap_count,
    'details': overlap_details,
}
overlap_path = OUTPUT_DIR / 's1_pcrtc_realattrs_spatial_leakage_check.json'
with overlap_path.open('w') as f:
    json.dump(overlap_result, f, indent=2)
print('Saved:', overlap_path)

Val patch 13279 overlaps 4 training patch(es)
Val patch 13184 overlaps 5 training patch(es)
Val patch 12009 overlaps 5 training patch(es)
Val patch 13333 overlaps 6 training patch(es)
Val patch 12597 overlaps 7 training patch(es)
Val patch 13075 overlaps 8 training patch(es)
Val patch 12477 overlaps 3 training patch(es)
Val patch 12265 overlaps 4 training patch(es)
Val patch 12933 overlaps 5 training patch(es)
Val patch 12320 overlaps 5 training patch(es)
Val patch 13510 overlaps 6 training patch(es)
Val patch 13282 overlaps 4 training patch(es)
Val patch 13274 overlaps 5 training patch(es)
Val patch 12955 overlaps 7 training patch(es)
Val patch 13069 overlaps 5 training patch(es)
Val patch 12172 overlaps 5 training patch(es)
Val patch 13588 overlaps 4 training patch(es)
Val patch 12681 overlaps 6 training patch(es)
Val patch 13235 overlaps 5 training patch(es)
Val patch 13286 overlaps 4 training patch(es)
Val patch 12909 overlaps 5 training patch(es)
Val patch 12944 overlaps 7 trainin

### How much overlap, not just whether

Every count above falls between 1 and 8 -- exactly the number of
neighbors a patch can have in a 2D grid, which is the signature of a
dense, overlapping sliding-window tiling scheme. Presence of overlap
isn't the same as severity, though: a 1% sliver and an 80% shared area
are very different problems. This computes the actual overlap fraction
for every val/train pair that touches at all.

In [ ]:
overlap_fractions = []
worst_cases = []
for pid, vbox in val_boxes:
    hits = tree.query(vbox)
    for h in hits:
        tbox = train_boxes[h]
        if tbox.intersects(vbox) and not tbox.touches(vbox):
            inter_area = tbox.intersection(vbox).area
            frac = inter_area / vbox.area  # fraction of the VALIDATION patch's own area that's shared
            overlap_fractions.append(frac)
            worst_cases.append((pid, frac))

overlap_fractions = np.array(overlap_fractions)
worst_cases.sort(key=lambda x: -x[1])

print(f'Overlap fraction stats across {len(overlap_fractions)} val/train pairs that touch:')
print(f'  mean:   {overlap_fractions.mean():.3f}')
print(f'  median: {np.median(overlap_fractions):.3f}')
print(f'  max:    {overlap_fractions.max():.3f}')
print(f'  >50% shared area: {(overlap_fractions > 0.5).sum()} pairs ({100 * (overlap_fractions > 0.5).mean():.1f}%)')
print(f'  >10% shared area: {(overlap_fractions > 0.1).sum()} pairs ({100 * (overlap_fractions > 0.1).mean():.1f}%)')
print()
print('Worst 10 val patches (highest single overlap fraction with any training patch):')
for pid, frac in worst_cases[:10]:
    print(f'  patch {pid}: {frac:.1%} of its area shared with a training patch')

severity_result = {
    'n_pairs': len(overlap_fractions),
    'mean_overlap_fraction': float(overlap_fractions.mean()),
    'median_overlap_fraction': float(np.median(overlap_fractions)),
    'max_overlap_fraction': float(overlap_fractions.max()),
    'pct_pairs_over_50pct': float(100 * (overlap_fractions > 0.5).mean()),
    'pct_pairs_over_10pct': float(100 * (overlap_fractions > 0.1).mean()),
}
severity_path = OUTPUT_DIR / 's1_pcrtc_realattrs_spatial_leakage_severity.json'
with severity_path.open('w') as f:
    json.dump(severity_result, f, indent=2)
print('\nSaved:', severity_path)

## Step 2 -- Unseen-date generalization test

Fetch Sentinel-1 (and Sentinel-2, for a visual check) from Microsoft
Planetary Computer for a date **one year after** the original LiDAR
survey -- same season/ice conditions as training, but an acquisition the
model has never touched in any form. Ground truth is unchanged: the same
LiDAR survey used everywhere else in this project, since terrain
roughness doesn't change on this timescale.

Runs on ~25 patches (not just a handful) so the result is a real metrics
table, comparable to `06`'s own validation numbers -- not just a
qualitative "looks fine" check.

In [7]:
import pystac_client
import planetary_computer
from dotenv import load_dotenv
from shapely.geometry import box as shapely_box, shape
from shapely.ops import unary_union
from concurrent.futures import ThreadPoolExecutor, as_completed

load_dotenv()
if os.environ.get('PC_SDK_SUBSCRIPTION_KEY'):
    planetary_computer.settings.set_subscription_key(os.environ['PC_SDK_SUBSCRIPTION_KEY'])

def aoi_from_lidar_patches(patches_dir, max_files=300, workers=8):
    from rasterio.warp import transform_geom
    paths = sorted(patches_dir.glob('lidar_patch_*.tif'))
    if len(paths) > max_files:
        stride = len(paths) / max_files
        paths = [paths[int(i * stride)] for i in range(max_files)]
    def read_bounds(path):
        with rasterio.open(path) as src:
            return src.crs, src.bounds
    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(read_bounds, p) for p in paths]
        for future in as_completed(futures):
            results.append(future.result())
    crs = results[0][0]
    native = unary_union([shapely_box(*bounds) for _, bounds in results])
    geojson = transform_geom(crs, 'EPSG:4326', native.__geo_interface__)
    return shape(geojson).buffer(0)

aoi = aoi_from_lidar_patches(LIDAR_DIR)
aoi_ll = aoi.convex_hull

NEW_WINDOW_CENTER = LIDAR_SURVEY_DATE.replace(year=LIDAR_SURVEY_DATE.year + 1)  # one year later -- same season
SEARCH_DAYS = 30
N_TEST_PATCHES = 25

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)
start = NEW_WINDOW_CENTER - dt.timedelta(days=SEARCH_DAYS)
end = NEW_WINDOW_CENTER + dt.timedelta(days=SEARCH_DAYS)

s1_items = sorted(catalog.search(
    collections=['sentinel-1-rtc'], intersects=aoi_ll.__geo_interface__,
    datetime=f'{start.isoformat()}/{end.isoformat()}',
).items(), key=lambda it: it.datetime)
print(f'New S1 scenes found: {len(s1_items)}')
assert len(s1_items) >= CONTEXT_K, f'Need at least {CONTEXT_K} new S1 scenes, found {len(s1_items)}'

# Sentinel-2, same window -- for Michel's requested visual check (no model needed,
# just the raw optical imagery next to the S1-based prediction)
s2_items = sorted(catalog.search(
    collections=['sentinel-2-l2a'], intersects=aoi_ll.__geo_interface__,
    datetime=f'{start.isoformat()}/{end.isoformat()}',
    query={'eo:cloud_cover': {'lt': 30}},
).items(), key=lambda it: it.properties.get('eo:cloud_cover', 100))
print(f'New S2 scenes found (cloud<30%): {len(s2_items)}')

# Explicit check -- not just eyeballing dates -- that these don't overlap the training acquisitions
existing_dates = set()
sample_attrs = S1_DIR / f's1_patch_{val_ids[0]}' / 'attrs.json'
if sample_attrs.exists():
    for a in json.load(open(sample_attrs)):
        if a.get('acquisition_date'):
            existing_dates.add(a['acquisition_date'])
print('Training acquisition dates:', sorted(existing_dates))

new_items = s1_items[:CONTEXT_K]
new_dates = {item.datetime.date().isoformat() for item in new_items}
print('Selected new S1 acquisition dates:', sorted(new_dates))
assert new_dates.isdisjoint(existing_dates), 'New dates overlap training dates -- pick a different window!'
print('Confirmed: no overlap with training acquisition dates.')

s2_item = s2_items[0] if s2_items else None
if s2_item:
    print('Using S2 scene:', s2_item.datetime.date(), '| cloud cover:', s2_item.properties.get('eo:cloud_cover'))
else:
    print('No usable S2 scene found in this window -- widen SEARCH_DAYS or relax cloud filter.')

New S1 scenes found: 9
New S2 scenes found (cloud<30%): 20
Training acquisition dates: ['2024-03-18', '2024-03-20', '2024-04-01', '2024-04-13', '2024-04-23', '2024-04-25', '2024-05-07']
Selected new S1 acquisition dates: ['2025-03-25', '2025-03-27', '2025-04-06']
Confirmed: no overlap with training acquisition dates.
Using S2 scene: 2025-05-15 | cloud cover: 0.120731


In [8]:
NEW_DIR = WORKING_REPO / 'raw_data' / 'tuk_unseen_date_test'
NEW_DIR.mkdir(parents=True, exist_ok=True)

merged_paths = []
for i, item in enumerate(new_items):
    with rasterio.open(item.assets['vv'].href) as vv_src:
        aoi_bounds = transform_bounds('EPSG:4326', vv_src.crs, *aoi_ll.bounds)
        window = from_bounds(*aoi_bounds, transform=vv_src.transform)
        vv = vv_src.read(1, window=window)
        out_transform = win_transform(window, vv_src.transform)
        out_crs = vv_src.crs
    with rasterio.open(item.assets['vh'].href) as vh_src:
        vh_window = from_bounds(*transform_bounds('EPSG:4326', vh_src.crs, *aoi_ll.bounds), transform=vh_src.transform)
        vh = vh_src.read(1, window=vh_window)
    h, w = min(vv.shape[0], vh.shape[0]), min(vv.shape[1], vh.shape[1])
    stacked = np.stack([vv[:h, :w], vh[:h, :w]]).astype(np.float32)
    out_path = NEW_DIR / f't{i}.tif'
    meta = {'driver': 'GTiff', 'count': 2, 'height': h, 'width': w, 'dtype': 'float32', 'crs': out_crs, 'transform': out_transform}
    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(stacked)
    merged_paths.append(str(out_path))
    print(f'Wrote t{i}.tif for {item.datetime.date()}')

Wrote t0.tif for 2025-03-25
Wrote t1.tif for 2025-03-27
Wrote t2.tif for 2025-04-06


In [9]:
S2_MOSAIC_PATH = NEW_DIR / 's2_visual_mosaic.tif'
if s2_item is not None:
    with rasterio.open(s2_item.assets['visual'].href) as src:
        aoi_bounds = transform_bounds('EPSG:4326', src.crs, *aoi_ll.bounds)
        window = from_bounds(*aoi_bounds, transform=src.transform)
        rgb = src.read([1, 2, 3], window=window)
        out_transform = win_transform(window, src.transform)
        meta = {'driver': 'GTiff', 'count': 3, 'height': rgb.shape[1], 'width': rgb.shape[2],
                'dtype': rgb.dtype, 'crs': src.crs, 'transform': out_transform}
        with rasterio.open(S2_MOSAIC_PATH, 'w', **meta) as dst:
            dst.write(rgb)
    print('Wrote S2 visual mosaic:', S2_MOSAIC_PATH)

Wrote S2 visual mosaic: /cs/student/project_msc/2025/aibh/jiayiche/raw_data/tuk_unseen_date_test/s2_visual_mosaic.tif


In [10]:
def extract_one_patch(patch_id, merged_paths, lidar_dir):
    lidar_path = lidar_dir / f'lidar_patch_{patch_id}.tif'
    with rasterio.open(lidar_path) as lsrc:
        lidar_bounds, lidar_crs = lsrc.bounds, lsrc.crs
        out_h, out_w = lsrc.height, lsrc.width
    views = []
    for mp in merged_paths:
        with rasterio.open(mp) as src:
            bounds = transform_bounds(lidar_crs, src.crs, *lidar_bounds, densify_pts=21)
            window = from_bounds(*bounds, transform=src.transform)
            patch = src.read(window=window, out_shape=(2, out_h, out_w))
        views.append(patch)
    return np.stack(views)  # [CONTEXT_K, 2, H, W]

test_patch_ids = random.Random(SEED + 2).sample(val_ids, min(N_TEST_PATCHES, len(val_ids)))
print(f'Testing on {len(test_patch_ids)} patches with genuinely new-date Sentinel-1 input')

new_raw = {pid: extract_one_patch(pid, merged_paths, LIDAR_DIR) for pid in test_patch_ids}

NameError: name 'test_patch_ids' is not defined

In [ ]:
def extract_s2_patch(patch_id, s2_mosaic_path, lidar_dir):
    lidar_path = lidar_dir / f'lidar_patch_{patch_id}.tif'
    with rasterio.open(lidar_path) as lsrc:
        lidar_bounds, lidar_crs = lsrc.bounds, lsrc.crs
        out_h, out_w = lsrc.height, lsrc.width
    with rasterio.open(s2_mosaic_path) as src:
        bounds = transform_bounds(lidar_crs, src.crs, *lidar_bounds, densify_pts=21)
        window = from_bounds(*bounds, transform=src.transform)
        rgb = src.read([1, 2, 3], window=window, out_shape=(3, out_h, out_w))
    return np.moveaxis(rgb, 0, -1)  # [H, W, 3] for imshow

new_s2_crops = {}
if S2_MOSAIC_PATH.exists():
    for pid in test_patch_ids[:6]:  # only need these for the example-patch visualization
        try:
            new_s2_crops[pid] = extract_s2_patch(pid, S2_MOSAIC_PATH, LIDAR_DIR)
        except Exception as exc:
            print(f'Could not extract S2 crop for patch {pid}: {exc}')
    print(f'Extracted S2 crops for {len(new_s2_crops)} example patches')
else:
    print('No S2 mosaic available -- skipping visual crops (S2 row will be blank in the figure).')

In [ ]:
def build_attrs_for_new_dates(items, context_k):
    vecs = []
    for item in items:
        age_days = (item.datetime.date() - LIDAR_SURVEY_DATE).days
        age_norm = age_days / 30.0
        orbit_dir = 1.0 if item.properties.get('sat:orbit_state') == 'ascending' else 0.0
        rel_orbit = (item.properties.get('sat:relative_orbit') or 0) / 175.0
        vecs.append([age_norm, orbit_dir, rel_orbit, 0.0, 0.0, 0.0, 0.0, 0.0])
    return torch.tensor(vecs, dtype=torch.float32).flatten()

new_attrs = build_attrs_for_new_dates(new_items, CONTEXT_K)

metric_rows = []
example_patches = []
model.eval()
with torch.no_grad():
    for pid, raw in new_raw.items():
        with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src:
            raw_lidar = src.read().astype(np.float32)
        target_arr = raw_lidar[0]
        mask = (raw_lidar[1] > 0.5) if raw_lidar.shape[0] > 1 else np.isfinite(target_arr)
        target_arr = np.nan_to_num(target_arr, nan=0.0, posinf=0.0, neginf=0.0)
        patch_mean = float(target_arr[mask].sum() / max(1, mask.sum()))
        target_demeaned = (target_arr - patch_mean) * mask

        views = []
        for k in range(CONTEXT_K):
            sar = np.nan_to_num(raw[k], nan=0.0, posinf=0.0, neginf=0.0)
            sar = np.maximum(sar, 1e-12)
            sar = 10.0 * np.log10(sar)
            sar_t = torch.from_numpy(sar).unsqueeze(0)
            sar_t = F.interpolate(sar_t, size=TARGET_HW, mode='bilinear', align_corners=False).squeeze(0)
            views.append(sar_t.repeat(2, 1, 1))
        condition = torch.cat(views, dim=0).unsqueeze(0).to(DEVICE)
        attrs = new_attrs.unsqueeze(0).to(DEVICE)
        target_t = torch.from_numpy(target_demeaned).unsqueeze(0).unsqueeze(0).to(DEVICE)
        mask_t = torch.from_numpy(mask).unsqueeze(0).to(DEVICE).bool()

        prediction = sampler(model, scheduler, target_t.shape, condition, attrs, DEVICE)
        mean_t = torch.tensor(patch_mean, device=DEVICE).view(1, 1, 1, 1)
        gt_absolute = target_t + mean_t
        pred_absolute = prediction + mean_t

        gt_i, pred_i, mask_i = gt_absolute[0], pred_absolute[0], mask_t[0]
        gt_valid = gt_i.squeeze()[mask_i].cpu().numpy()
        pred_valid = pred_i.squeeze()[mask_i].cpu().numpy()
        metric_rows.append({
            'patch_id': pid,
            'rmse_m': float(rmse(gt_i, pred_i, mask_i).item()),
            'bias_m': float(bias(gt_i, pred_i, mask_i).item()),
            'sigma_error_pct': float(sigma_error(gt_i, pred_i, mask_i).item()),
            'normal_angle_error_deg': float(normal_angle_error(gt_i, pred_i, mask_i, pixel_size=1.0, degrees=True).item()),
            'jsd': float(average_jsd_multiscale(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
            'psd_rmse': float(log_psd_rmse(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
            'zncc': float(zncc(gt_i, pred_i, mask_i).item()),
            'gt_std_val': float(gt_valid.std()) if gt_valid.size > 0 else float('nan'),
            'pred_std_val': float(pred_valid.std()) if pred_valid.size > 0 else float('nan'),
        })
        if len(example_patches) < 6:
            example_patches.append({'patch_id': pid, 'gt': gt_i.squeeze().cpu().numpy(),
                                     'pred': pred_i.squeeze().cpu().numpy(), 'mask': mask_i.squeeze().cpu().numpy()})

metrics_path = OUTPUT_DIR / 's1_pcrtc_realattrs_unseen_date_metrics.json'
with metrics_path.open('w') as f:
    json.dump(metric_rows, f, indent=2)
print('Saved:', metrics_path)
mean_metrics = {k: float(np.nanmean([r[k] for r in metric_rows])) for k in metric_rows[0] if k != 'patch_id'}
print('Mean metrics on unseen-date imagery:', mean_metrics)

### Compare against `06`'s original validation metrics

Paste `06`'s original mean metrics below (from
`s1_pcrtc_realattrs_validation_metrics.json`) to see side by side whether
performance holds up on genuinely new-date imagery, or drops off.

In [ ]:
original_metrics_path = OUTPUT_DIR / 's1_pcrtc_realattrs_validation_metrics.json'
if original_metrics_path.exists():
    orig_rows = json.load(open(original_metrics_path))
    orig_mean = {k: float(np.nanmean([r[k] for r in orig_rows])) for k in orig_rows[0] if k != 'patch_id'}
    print(f'{"metric":<20}{"original val":>15}{"unseen date":>15}')
    for k in mean_metrics:
        if k in orig_mean:
            print(f'{k:<20}{orig_mean[k]:>15.4f}{mean_metrics[k]:>15.4f}')
else:
    print('Original validation metrics file not found -- compare manually against the numbers already reported for 06.')

In [ ]:
n_show = min(6, len(example_patches))
fig, axes = plt.subplots(4, n_show, figsize=(4 * n_show, 16), squeeze=False)
for col, ex in enumerate(example_patches[:n_show]):
    gt_c = ex['gt'] - ex['gt'][ex['mask']].mean()
    pred_c = ex['pred'] - ex['pred'][ex['mask']].mean()
    pid = ex['patch_id']
    axes[0, col].set_title(f"Patch {pid}", fontweight='bold')
    if pid in new_s2_crops:
        axes[0, col].imshow(new_s2_crops[pid])
    axes[0, col].axis('off')
    axes[1, col].imshow(gt_c, cmap='RdBu_r'); axes[1, col].axis('off')
    axes[2, col].imshow(pred_c, cmap='RdBu_r'); axes[2, col].axis('off')
    axes[3, col].imshow(pred_c - gt_c, cmap='seismic'); axes[3, col].axis('off')
for row, label in enumerate(['New S2 imagery (visual check)', 'GT LiDAR (centered)', 'Pred, new S1 date (centered)', 'Error']):
    axes[row, 0].text(-0.25, 0.5, label, ha='right', va='center', transform=axes[row, 0].transAxes, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 's1_pcrtc_realattrs_unseen_date_reconstructions.png', dpi=150, bbox_inches='tight')
print('Saved:', OUTPUT_DIR / 's1_pcrtc_realattrs_unseen_date_reconstructions.png')
plt.show()

## Notes

- **Sentinel-2 visual comparison**: Michel confirmed he doesn't have
  Tessa's checkpoint and is asking her directly. For now, his actual
  request is simpler than a full model comparison -- just the raw S2
  imagery displayed alongside the S1-based prediction (built into the
  visualization above), not a Tessa-model prediction. Revisit with a real
  S2 model comparison if the checkpoint comes through later.
- **Extending this to the raw-SAFE track**: this notebook only checks
  `pcrtc/06`. If time allows, the same two steps could be run against the
  best raw-SAFE checkpoint too, since Michel's trust question applies
  equally there -- lower priority than `06` since it isn't the headline
  result, but worth a mention if he asks about the whole study rather
  than just the best config.